# Chuyên đề 07 · MOSS-TTS: từ âm thanh thô đến mô hình đọc: dựng lại từng bước nhỏ của báo cáo kỹ thuật

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd07-moss-tts.ipynb)

Notebook đi kèm [Chuyên đề 07 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/moss-tts/). Chuyên đề 05 dựng lại *bộ token hóa* âm thanh. Nhưng
muốn có một mô hình đọc văn bản thành giọng nói (TTS), ta còn cần **dữ liệu**: hàng triệu giờ cặp (âm thanh, lời),
mỗi cặp đúng một người nói, và một **mô hình sinh** đọc được 32 lớp token ấy. Báo cáo kỹ thuật
[*MOSS-TTS*](https://arxiv.org/abs/2603.18090) (SII-OpenMOSS, 2026) kể cả quy trình: bốn tầng lọc âm thanh thô lấy
từ mạng, dữ liệu tổng hợp cho những gì mạng không có, bộ token hóa MOSS-Audio-Tokenizer, cách xếp 33 kênh token cho
một Transformer tự hồi quy, và bốn pha tiền huấn luyện.

Ta không có hàng triệu giờ âm thanh hay một mô hình 8 tỉ tham số. Nhưng phần lớn quy trình là những **phép tính
nhỏ, viết rõ trong báo cáo**: một công thức độ lợi, ba luật lọc lời, một phép chọn giọng mẫu bằng cosin, một phép
dịch các lớp token. Ta tự viết từng phép ấy và chạy nó trên dữ liệu thật: 3.000 bản ghi chữ số tiếng Anh của
Chuyên đề 05, ghép thành một "cuộc trò chuyện" giữa ba người.

**Bạn sẽ làm:**
1. Chuẩn hóa âm lượng theo công thức của tầng 1, và thấy một câu trong báo cáo có hai cách đọc cho hai kết quả rất
   khác nhau.
2. Dựng một bản ghi dài nhiều người nói, tự viết một bộ tách người nói nhỏ (dò tiếng bằng năng lượng, gom cụm
   vectơ giọng), rồi gộp đoạn theo luật của tầng 2.
3. Chép lời bằng một "người chấm" chữ số, chạy ba luật lọc và phép kiểm một người nói của tầng 3, rồi tốc độ ký tự
   của tầng 4.
4. Chọn giọng mẫu cho từng đoạn: 5 mẩu cắt từ mỗi đoạn khác của cùng người, chấm bằng cosin của vectơ giọng.
5. Xếp 33 kênh token theo delay pattern, kiểm tra phép dịch ngược, đếm bước so với Local Transformer, rồi chạy lịch
   tốc độ học bốn pha trên một bài toán nhỏ.

**Cần biết trước:** [Bài 14 · Token & embedding](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-embedding/),
[Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/),
[Bài 22 · Tiền huấn luyện & tinh chỉnh](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/tien-huan-luyen/) và
[Chuyên đề 05 · Token hóa âm thanh](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-hoa-am-thanh/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải dữ liệu lần đầu.

Notebook dùng **PyTorch**, như Chuyên đề 05. Nó mượn của chuyên đề ấy hai "người chấm" đã huấn luyện sẵn (một CNN
nhỏ nhận ra chữ số, một CNN nhận ra giọng) và vài chuỗi token thật; mọi phép của MOSS-TTS thì ta tự viết và chạy
trực tiếp. Mọi con số của báo cáo được ghi kèm số trang (bản v2 trên arXiv); mọi con số khác do các ô bên dưới tính.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

In [ ]:
import base64
import hashlib
import io
import json
import math
import os
import re
import tarfile
import urllib.request
import wave
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import Audio, display
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=1):
    return vn(100 * p, digits) + "%"


def sci(x, digits=1):
    """Số rất nhỏ hoặc rất lớn: 2·10⁻⁴ (0 in ra 0; bỏ “,0” thừa)."""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)) + 1e-9)
    m = vn(x / 10**e, digits).removesuffix(",0")
    return f"{m}·10{str(e).translate(SUP)}"


def vn_tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.001 <= v <= 10000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Nhãn trục kiểu Việt Nam; gọi ngay trước plt.show(). Trục log ngắn lấy vạch 1–2–5, trục log dài lấy lũy thừa của 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def play(x, label=""):
    """Phát một đoạn âm thanh 8 kHz ngay trong notebook (giữ nguyên độ to, cắt về [−1, 1])."""
    if label:
        print(label)
    display(Audio(np.clip(np.asarray(x, np.float64), -1, 1), rate=SR, normalize=False))

## 2 · Dữ liệu: 3.000 chữ số được đọc to, và hai người chấm của Chuyên đề 05

[Free Spoken Digit Dataset](https://github.com/Jakobovski/free-spoken-digit-dataset) (FSDD, phiên bản 1.0.10, giấy
phép CC BY-SA 4.0): 6 người, mỗi người đọc mỗi chữ số từ 0 đến 9 năm mươi lần, thu ở 8.000 mẫu mỗi giây, mỗi mẫu
16 bit. Như ở Chuyên đề 05, ta kiểm MD5 của *nội dung* 3.000 tệp wav. Khác Chuyên đề 05, lần này ta **giữ nguyên độ
to gốc** của từng bản ghi, vì tầng 1 của báo cáo chính là bước chỉnh độ to.

Tệp `codec-weights.json` của Chuyên đề 05 chứa trọng số (int8) của hai CNN nhỏ, huấn luyện trên các bản ghi có số thứ
tự từ 10 trở lên: một CNN đoán chữ số, một CNN đoán người đọc. Ở đây chúng đóng vai mô hình chép lời (ASR) và mô hình
vectơ giọng của báo cáo. Tệp cũng có vài chuỗi token 32 lớp thật, dùng ở mục 10.

In [ ]:
SR = 8000
SPEAKERS = ("george", "jackson", "lucas", "nicolas", "theo", "yweweler")
WORDS = ("zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine")
FSDD = "https://github.com/Jakobovski/free-spoken-digit-dataset/archive/refs/tags/v1.0.10.tar.gz"
TGZ = Path("fsdd-v1.0.10.tar.gz")
if not TGZ.exists():
    urllib.request.urlretrieve(FSDD, TGZ)


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


W = fetch_json("codec-weights.json", "6f4b0e5508a869f770f628dd0468e742")

In [ ]:
raw = {}
with tarfile.open(TGZ) as tf:
    for m in tf.getmembers():
        if m.isfile() and m.name.endswith(".wav") and "/recordings/" in m.name:
            raw[m.name.rsplit("/", 1)[1]] = tf.extractfile(m).read()
h = hashlib.md5()
for k in sorted(raw):
    h.update(raw[k])
assert len(raw) == 3000 and h.hexdigest() == "2c93442f2c808e24d4d913f6be24d32c", "FSDD bị hỏng, hãy xóa và tải lại"

waves, digit, spk, idx = [], [], [], []
for name in sorted(raw):
    d, s, i = name[:-4].split("_")
    with wave.open(io.BytesIO(raw[name])) as w:
        assert (w.getnchannels(), w.getsampwidth(), w.getframerate()) == (1, 2, SR)
        waves.append(np.frombuffer(w.readframes(w.getnframes()), "<i2").astype(np.float64) / 32768)
    digit.append(int(d))
    spk.append(SPEAKERS.index(s))
    idx.append(int(i))
digit, spk, idx = np.array(digit), np.array(spk), np.array(idx)
dur = np.array([len(x) for x in waves]) / SR
print(f"{vn(len(waves), 0)} bản ghi, {vn(dur.sum() / 60, 1)} phút; dài {vn(dur.min(), 2)}–{vn(dur.max(), 2)} s")

In [ ]:
def hz_to_mel(f):
    return 2595 * np.log10(1 + np.asarray(f, np.float64) / 700)


def mel_filters(n_fft, n_mels):
    """Các bộ lọc tam giác trên thang mel, (n_fft/2 + 1, n_mels), như Chuyên đề 05."""
    freqs = np.linspace(0, SR / 2, n_fft // 2 + 1)
    edges = 700 * (10 ** (np.linspace(hz_to_mel(0), hz_to_mel(SR / 2), n_mels + 2) / 2595) - 1)
    lo, mid, hi = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    tri = np.maximum(0, np.minimum((freqs[None] - lo) / (mid - lo), (hi - freqs[None]) / (hi - mid)))
    return torch.tensor((tri * (2 / (hi - lo))).T, dtype=torch.float32)


HANN, FB40 = torch.hann_window(256), mel_filters(256, 40)


def logmel(x):
    """(B, L) → (B, 40, khung): log₁₀ độ lớn trong 40 dải mel; cửa sổ 32 ms, bước 10 ms."""
    s = torch.stft(x, 256, 80, window=HANN, center=True, return_complex=True).abs()
    return (s.transpose(1, 2) @ FB40).clamp_min(1e-5).log10().transpose(1, 2)


class Judge(nn.Module):
    """Người chấm của Chuyên đề 05: CNN 1-D trên 40 dải log-mel → vectơ 128 chiều → lớp."""

    def __init__(self, classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(40, 128, 5, padding=2), nn.ReLU(),
            nn.Conv1d(128, 128, 5, stride=2, padding=2), nn.ReLU(),
            nn.Conv1d(128, 128, 5, stride=2, padding=2), nn.ReLU(),
        )  # fmt: skip
        self.emb, self.head = nn.Linear(256, 128), nn.Linear(128, classes)

    def embed(self, x):
        h = self.conv((logmel(x) + 5) / 2)
        return F.relu(self.emb(torch.cat([h.mean(-1), h.amax(-1)], 1)))

    def forward(self, x):
        return self.head(self.embed(x))


def unpack(p):
    """Giải nén trọng số: int8 × hệ số tỉ lệ float16 của từng hàng, hoặc float16."""
    sd = {}
    for k, v in p.items():
        if "q" in v:
            q = np.frombuffer(base64.b64decode(v["q"]), np.int8).astype(np.float32).reshape(v["shape"][0], -1)
            s = np.frombuffer(base64.b64decode(v["s"]), "<f2").astype(np.float32)
            sd[k] = torch.tensor((q * s[:, None]).reshape(v["shape"]))
        else:
            sd[k] = torch.tensor(
                np.frombuffer(base64.b64decode(v["f16"]), "<f2").astype(np.float32).reshape(v["shape"])
            )
    return sd


asr, voice = Judge(10), Judge(len(SPEAKERS))
asr.load_state_dict(unpack(W["judgeDigit"]))
voice.load_state_dict(unpack(W["judgeSpk"]))
asr.eval(), voice.eval()


def judge_input(xs):
    """Như lúc hai người chấm học: mỗi đoạn chỉnh về RMS 0,1, đệm số 0 đến ít nhất 1,28 s."""
    n = max(10240, max(len(x) for x in xs))
    out = np.zeros((len(xs), n), np.float32)
    for b, x in enumerate(xs):
        out[b, : len(x)] = x * 0.1 / np.sqrt(np.mean(x**2) + 1e-12)
    return torch.tensor(np.clip(out, -1, 1))


TEST = np.where(idx < 5)[0]  # các bản ghi hai người chấm chưa từng thấy
with torch.no_grad():
    Xt = judge_input([waves[c] for c in TEST])
    acc_asr = float((asr(Xt).argmax(-1).numpy() == digit[TEST]).mean())
    acc_voice = float((voice(Xt).argmax(-1).numpy() == spk[TEST]).mean())
print(f"trên {len(TEST)} bản ghi kiểm tra: đoán đúng chữ số {pct(acc_asr)}, đúng người đọc {pct(acc_voice)}")

## 3 · Báo cáo làm gì, và ta dựng lại gì

Báo cáo chia quy trình dữ liệu làm ba giai đoạn (mục 5.1, tr. 10–14), rồi đến bộ token hóa và mô hình:

| Bước | Báo cáo (trang) | Ở đây |
|---|---|---|
| Tầng 1 · làm sạch | khử nhiễu MossFormer2-SE-48K, ghi FLAC, độ lợi clip(−20 − L, −3, 3) dB rồi chia cho max \|x\| (tr. 11) | công thức độ lợi, theo hai cách đọc (mục 4) |
| Tầng 2 · người nói | DiariZen tách người nói; bỏ mẩu < 0,1 s, gộp đoạn liền nhau cùng nhãn, tối đa 1 giờ (tr. 11–12) | dò tiếng bằng năng lượng, gom cụm vectơ giọng, đúng luật gộp (mục 5) |
| Tầng 3 · lời | MOSS-Transcribe-Diarize chép lời; 3 luật, một LLM chẩn đoán và sửa, kiểm chỉ [S1] (tr. 12–13) | người chấm chữ số chép lời; 3 luật, phần dễ của LLM, kiểm [S1] (mục 6) |
| Tầng 4 · chung | DNSMOS > 2,8, AudioBox PQ > 6,5, Whisper và LLM cùng nhãn ngôn ngữ, tốc độ ký tự (tr. 13) | tốc độ ký tự (mục 7) |
| Tổng hợp | giọng mẫu: 5 mẩu từ mỗi đoạn khác, cosin vectơ giọng WavLM; lời nhiễu, phiên âm; số token đích (tr. 13–14) | giọng mẫu và số token đích (mục 8) |
| Bộ token hóa | MOSS-Audio-Tokenizer: 24 kHz → 12,5 khung/s × 32 lớp (tr. 4–7) | đếm (mục 9); Chuyên đề 05 đã dựng lại |
| Mô hình | 33 kênh; delay pattern (8 tỉ) hay Local Transformer (1,7 tỉ); mất mát có trọng số λ (tr. 7–10) | xếp token thật, đếm bước, mất mát (mục 10) |
| Tiền huấn luyện | bốn pha, khởi động–giữ–giảm, 2·10⁻⁴ → 2·10⁻⁶ (tr. 15–16) | cùng hình dạng lịch trên một bài toán nhỏ (mục 11) |

Những gì cần mô hình lớn (khử nhiễu, chấm chất lượng, LLM, Whisper) ta không chạy; ở mỗi chỗ ấy notebook nói rõ.

## 4 · Tầng 1: chuẩn hóa âm lượng

Âm thanh lấy từ mạng to nhỏ chênh nhau hàng chục dB. Báo cáo đo **mức RMS** của cả bản ghi theo dBFS, tức decibel so
với biên ±1: một sóng vuông chạm biên ở 0 dBFS, một sóng sin chạm biên ở khoảng −3 dBFS, và tín hiệu càng nhỏ thì mức
càng âm. Rồi nó nhân bản ghi với một độ lợi bị chặn (tr. 11):

$$L = 20\log_{10}\sqrt{\tfrac{1}{T}\textstyle\sum_t x_t^2 + \epsilon}, \qquad g = \mathrm{clip}(-20 - L,\ -3,\ 3)\ \text{dB}, \qquad x \leftarrow x \cdot 10^{g/20}$$

Mục tiêu là −20 dBFS, nhưng độ lợi không vượt quá ±3 dB: một bản rất nhỏ chỉ được nâng 3 dB chứ không bị kéo lên
thẳng −20, và một bản đã to chỉ bị hạ 3 dB. Sau đó là bước thứ hai: "chia cho giá trị tuyệt đối lớn nhất của mẫu,
để đưa kết quả về khoảng [−1, 1]".

In [ ]:
EPS = 1e-10  # ε trong căn; báo cáo không nêu giá trị


def level_dbfs(x):
    return 20 * np.log10(np.sqrt(np.mean(np.square(x)) + EPS))


def gain_db(L, target=-20.0, clamp=3.0):
    return np.clip(target - L, -clamp, clamp)


for L in (-32, -21, -14):
    print(f"L = {vn(L, 0)} dBFS → lẽ ra cần {vn(-20 - L, 0):>3s} dB, được {vn(gain_db(L), 0)} dB")
assert [gain_db(L) for L in (-32, -21, -14)] == [3, 1, -3]

L0 = np.array([level_dbfs(x) for x in waves])
print(f"\nFSDD gốc: mức RMS từ {vn(L0.min(), 1)} đến {vn(L0.max(), 1)} dBFS, trung vị {vn(np.median(L0), 1)}")
for s, name in enumerate(SPEAKERS):
    print(f"  {name:9s} trung vị {vn(np.median(L0[spk == s]), 1):>6s} dBFS")
print(f"{pct(np.mean(np.abs(-20 - L0) > 3))} số bản ghi cần nhiều hơn ±3 dB, nên bị chặn")

Sáu người của FSDD thu ở những mức rất khác nhau: theo và yweweler nhỏ hơn bốn người kia khoảng 12–22 dB. Phần lớn
bản ghi cần nhiều hơn 3 dB, nên độ lợi bị chặn ở hầu hết chúng.

Bước thứ hai có **hai cách đọc**. Đọc theo từng chữ (A): *luôn* chia cho max |x|. Nhưng khi ấy x · 10^{g/20} chia
cho max |x · 10^{g/20}| đúng bằng x / max |x|: độ lợi ở bước một bị triệt tiêu hoàn toàn, và mức RMS sau cùng chỉ
còn phụ thuộc hình dạng sóng (tỉ số giữa đỉnh và RMS). Cách đọc thứ hai (B): chỉ chia khi có mẫu vượt ±1, tức một
bước chống tràn. Báo cáo không nói rõ cách nào.

### 🤔 Dự đoán trước
Độ chênh giữa bản to nhất và bản nhỏ nhất của FSDD là khoảng 38 dB. Sau tầng 1, theo cách đọc B, còn bao nhiêu?
Theo cách đọc A thì sao?

In [ ]:
def stage1(x, always_peak):
    """Tầng 1: độ lợi bị chặn, rồi chia cho max|x| (A: luôn luôn; B: chỉ khi vượt ±1)."""
    y = x * 10 ** (gain_db(level_dbfs(x)) / 20)
    peak = np.abs(y).max()
    return y / peak if always_peak or peak > 1 else y


ex = waves[0]
assert np.allclose(stage1(ex, True), ex / np.abs(ex).max()), "cách đọc A: độ lợi bị triệt tiêu"
LA = np.array([level_dbfs(stage1(x, True)) for x in waves])
LB = np.array([level_dbfs(stage1(x, False)) for x in waves])
n_over = sum(np.abs(x * 10 ** (gain_db(level_dbfs(x)) / 20)).max() > 1 for x in waves)
for name, v in (("gốc", L0), ("cách đọc B", LB), ("cách đọc A", LA)):
    lo, hi = np.percentile(v, [5, 95])
    print(f"{name:11s} chênh to nhất − nhỏ nhất {vn(v.max() - v.min(), 1):>5s} dB · "
          f"5%–95%: {vn(lo, 1)} … {vn(hi, 1)} dBFS")  # fmt: skip
print(f"cách đọc B chỉ chia ở {vn(n_over, 0)} bản ghi (có mẫu vượt ±1 sau độ lợi)")

Hình dưới vẽ mức RMS của từng người (mỗi hộp là 500 bản ghi), ở bản gốc, sau cách đọc B và sau cách đọc A.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4), sharey=True)
for ax, (title, v, col, hatch) in zip(
    axes,
    (
        ("bản gốc", L0, GREY, ""),
        ("cách đọc B: chỉ chống tràn", LB, BLUE, "//"),
        ("cách đọc A: luôn chia max|x|", LA, ORANGE, ".."),
    ),
    strict=True,
):
    bp = ax.boxplot([v[spk == s] for s in range(6)], patch_artist=True, widths=0.6, showfliers=False)
    for b in bp["boxes"]:
        b.set(facecolor=col, alpha=0.55, hatch=hatch)
    ax.axhline(-20, color="k", lw=0.8, ls="--")
    ax.set_xticks(range(1, 7), SPEAKERS, rotation=45, fontsize=8)
    ax.set_title(title, fontsize=10)
axes[0].set_ylabel("mức RMS L (dBFS)")
axes[0].text(6.45, -21, "mục tiêu −20", ha="right", va="top", fontsize=8)
vn_axes(fig)
plt.tight_layout()
plt.show()

Theo cách đọc B, độ chênh chỉ giảm từ 37,9 xuống 31,9 dB: ±3 dB là quá ít so với khoảng cách 12–22 dB giữa các người,
nên theo và yweweler vẫn nhỏ hơn hẳn. Theo cách đọc A, mọi bản ghi có đỉnh bằng 1, và mức RMS dồn về một dải hẹp hơn
nhiều (độ chênh 16,8 dB); nhưng khi ấy công thức độ lợi ±3 dB thành thừa. Báo cáo viết rằng khoảng ±3 dB để "không
nén quá tay bản đã nhỏ và không khuếch đại quá đà bản bất thường", một mục đích chỉ có nghĩa ở cách đọc B. Mã nguồn
của khâu này không được công bố, nên ta không biết chắc.

Bài học chung: một câu mô tả quy trình bằng lời có thể che một lựa chọn quan trọng. Muốn tái lập, hãy viết từng câu
thành mã rồi kiểm xem nó có triệt tiêu chính nó không.

## 5 · Tầng 2: tách người nói, rồi gộp đoạn

Một bản ghi trên mạng (một tập podcast, một bộ phim) có nhiều người nói. Bộ tách người nói (*speaker diarization*)
trả về một chuỗi khoảng thời gian, mỗi khoảng một nhãn người nói, và **nhãn chỉ có nghĩa trong bản ghi ấy**:
SPEAKER_00 của tập này không phải SPEAKER_00 của tập khác (tr. 11). Báo cáo dùng DiariZen. Ta dựng một bộ nhỏ theo
cùng ý: dò những khoảng có tiếng, tính vectơ giọng của từng khoảng, rồi gom các khoảng có vectơ giống nhau thành một
người.

Trước hết là một "cuộc trò chuyện": ba người (george, lucas, theo) thay phiên nhau 12 lượt, mỗi lượt 2–4 chữ số lấy từ
các bản ghi kiểm tra, cách nhau những khoảng lặng 0,15–0,45 s trong một lượt và 0,5–0,9 s giữa hai lượt. Ba lượt có
thêm một tiếng "tách" dài 40 ms (như tiếng thở hay tiếng động). Mọi bản ghi được chỉnh về cùng mức RMS, như khi ba
người dùng chung một micro, cộng một lớp nhiễu nền rất nhỏ (−60 dBFS).

In [ ]:
CAST = ("george", "lucas", "theo")
ORDER = (0, 1, 0, 2, 1, 2, 0, 1, 2, 0, 2, 1)  # người nói của 12 lượt
CLICK_TURNS = (2, 6, 9)  # các lượt có một tiếng "tách" giữa hai chữ số đầu
rng = np.random.default_rng(0)
pool = {who: list(rng.permutation(TEST[spk[TEST] == SPEAKERS.index(name)])) for who, name in enumerate(CAST)}
parts, truth, clicks = [np.zeros(int(0.3 * SR))], [], []  # truth: (người, bắt đầu, kết thúc, bản ghi)


def now():
    return sum(len(p) for p in parts) / SR


for turn, who in enumerate(ORDER):
    n = int(rng.integers(2, 5))
    for c in range(n):
        clip = pool[who].pop()
        truth.append((who, now(), now() + len(waves[clip]) / SR, clip))
        parts.append(waves[clip] * 0.05 / np.sqrt(np.mean(waves[clip] ** 2)))
        if c == n - 1:
            break
        gap = int(rng.uniform(0.15, 0.45) * SR)
        if turn in CLICK_TURNS and c == 0:
            parts.append(np.zeros(gap // 2))
            clicks.append(now())
            parts.append(rng.normal(0, 0.04, 320) * np.hanning(320))  # 40 ms
            parts.append(np.zeros(gap - gap // 2))
        else:
            parts.append(np.zeros(gap))
    parts.append(np.zeros(int(rng.uniform(0.5, 0.9) * SR)))
rec = np.concatenate(parts)
rec = stage1(rec + rng.normal(0, 1e-3, len(rec)), always_peak=False)
print(f"bản ghi dài {vn(len(rec) / SR, 1)} s: {len(ORDER)} lượt, {len(truth)} chữ số, {len(clicks)} tiếng tách")
play(rec, "cuộc trò chuyện của ba người")

**Dò tiếng bằng năng lượng.** Cắt bản ghi thành khung 10 ms, tính năng lượng của từng khung theo dB; khung nào cách
khung to nhất không quá 35 dB là "có tiếng". Các khung có tiếng liền nhau thành một khoảng, và hai khoảng cách nhau
dưới 80 ms được nối lại (khoảng ngắt bên trong một từ). Đây là bộ dò tiếng nói (VAD) đơn giản nhất; báo cáo khử
nhiễu trước tầng 2 chính là để bước này chính xác hơn.

In [ ]:
def vad(x, frame=80, floor_db=35, bridge=0.08):
    n = len(x) // frame
    e = 10 * np.log10(np.mean(x[: n * frame].reshape(n, frame) ** 2, 1) + 1e-12)
    on = np.concatenate([[False], e > e.max() - floor_db, [False]])
    starts, ends = np.where(on[1:] & ~on[:-1])[0], np.where(~on[1:] & on[:-1])[0]
    out = []
    for a, b in zip(starts * frame / SR, ends * frame / SR, strict=True):
        if out and a - out[-1][1] < bridge:
            out[-1][1] = b
        else:
            out.append([a, b])
    return [tuple(p) for p in out]


spans = vad(rec)
lens = np.array([b - a for a, b in spans])
print(
    f"{len(spans)} khoảng có tiếng; {int((lens < 0.1).sum())} khoảng ngắn hơn 0,1 s (ngắn nhất {vn(lens.min() * 1000, 0)} ms)"
)

**Gom cụm vectơ giọng.** Người chấm giọng của Chuyên đề 05 cho mỗi khoảng một vectơ 128 chiều. Ta *không* dùng đầu
ra 6 lớp của nó: một bộ tách người nói thật không biết trước tên ai, nó chỉ hỏi "hai khoảng này có cùng một giọng
không". Gom cụm từ dưới lên: mỗi khoảng là một cụm, rồi lặp lại việc nhập hai cụm có cosin trung bình cao nhất, cho
đến khi cosin ấy dưới một ngưỡng θ = 0,9. Cụm được đặt tên theo thứ tự xuất hiện.

In [ ]:
def embed(xs):
    with torch.no_grad():
        return F.normalize(voice.embed(judge_input(xs)), dim=1).numpy()


def cluster(E, theta):
    """Gom cụm từ dưới lên, liên kết trung bình trên cosin; trả về nhãn theo thứ tự xuất hiện."""
    S = E @ E.T
    groups = [[i] for i in range(len(E))]
    while len(groups) > 1:
        sims = {(a, b): S[np.ix_(groups[a], groups[b])].mean() for a in range(len(groups)) for b in range(a)}
        (a, b), best = max(sims.items(), key=lambda kv: kv[1])
        if best < theta:
            break
        groups[b] += groups.pop(a)
    lab = np.empty(len(E), int)
    for k, g in enumerate(sorted(groups, key=min)):
        lab[g] = k
    return lab


def who_said(a, b):
    """Người nói thật có nhiều thời gian nhất trong [a, b] (−1 nếu không ai: tiếng tách)."""
    ov = [(min(e, b) - max(s, a), who) for who, s, e, _ in truth]
    best = max(ov)
    return best[1] if best[0] > 0 else -1


span_x = [rec[int(a * SR) : int(b * SR)] for a, b in spans]
E_spans = embed(span_x)
THETA = 0.9
lab = cluster(E_spans, THETA)
D = [(f"SPEAKER_{k:02d}", a, b) for k, (a, b) in zip(lab, spans, strict=True)]
true_of = np.array([who_said(a, b) for a, b in spans])
ok = lens >= 0.1
for k in sorted(set(lab)):
    m = lab == k
    names = sorted({CAST[w] if w >= 0 else "tiếng tách" for w in true_of[m & ok]}) or ["chỉ có mẩu < 0,1 s"]
    print(f"SPEAKER_{k:02d}: {int(m.sum()):2d} khoảng ({int((m & ~ok).sum())} khoảng < 0,1 s); người nói thật: "
          f"{', '.join(names)}")  # fmt: skip
with torch.no_grad():
    named = voice(judge_input(span_x)).argmax(-1).numpy()
hit = np.mean([SPEAKERS[n] == CAST[w] for n, w in zip(named[ok], true_of[ok], strict=True)])
print(f"\nso sánh: nếu gọi tên từng khoảng bằng đầu ra 6 người, chỉ đúng {pct(hit)} số khoảng dài ≥ 0,1 s")

Ba người, ba cụm, và không cụm nào lẫn hai người. Cụm thứ tư, SPEAKER_02, chỉ gồm 7 mẩu ngắn hơn 0,1 s: ba tiếng tách
và vài mảnh vụn ở đầu hay cuối một từ. Vectơ giọng của chúng không giống ai, nên chúng đứng riêng; tầng 2 sẽ bỏ chúng
đi. Đáng chú ý là dòng cuối: cùng một mạng, nếu dùng thẳng đầu ra "đây là
ai trong 6 người" thì sai ở một phần đáng kể số khoảng (nó hay nhầm lucas với yweweler), còn câu hỏi "hai khoảng này
có cùng giọng không" thì dễ hơn nhiều. Đó cũng là lý do nhãn của báo cáo chỉ có nghĩa trong một bản ghi.

**Gộp đoạn** (tr. 11–12): bỏ mọi mẩu ngắn hơn τ_min = 0,1 s, rồi duyệt theo thời gian và gộp mọi đoạn liền nhau có cùng
nhãn, *không xét khoảng lặng giữa chúng* (khoảng lặng giữa hai đoạn cùng người là chỗ ngừng tự nhiên trong một lượt
nói). Cuối cùng chặn ở một giờ kể từ đầu đoạn đầu tiên: đoạn cuối bị cắt ở mốc ấy, và bỏ nếu còn ngắn hơn τ_min.

### 🤔 Dự đoán trước
Thứ tự hai việc "bỏ mẩu ngắn" và "gộp đoạn" có quan trọng không? Nếu gộp trước rồi mới bỏ mẩu ngắn, cuộc trò chuyện 12
lượt của ta còn bao nhiêu đoạn?

In [ ]:
def consolidate(D, tau_min=0.1, cap=3600.0, drop_first=True):
    """Tầng 2: bỏ mẩu < τ_min, gộp các đoạn liền nhau cùng nhãn, chặn ở cap giây từ đầu đoạn đầu tiên."""
    seq = [d for d in D if d[2] - d[1] >= tau_min] if drop_first else D
    A = []
    for k, a, b in seq:
        if A and A[-1][0] == k:
            A[-1][2] = b
        else:
            A.append([k, a, b])
    if not drop_first:
        A = [x for x in A if x[2] - x[1] >= tau_min]
    t_lim = A[0][1] + cap
    out = []
    for k, a, b in A:
        if a >= t_lim:
            break
        b = min(b, t_lim)
        if b - a >= tau_min:
            out.append((k, a, b))
    return out


SEGS = consolidate(D)
alt = consolidate(D, drop_first=False)
print(f"{len(spans)} khoảng → bỏ mẩu ngắn rồi gộp: {len(SEGS)} đoạn · gộp trước rồi bỏ: {len(alt)} đoạn "
      f"(cuộc trò chuyện có {len(ORDER)} lượt)")  # fmt: skip
pure = [len({w for w in true_of[(np.array([a for a, _ in spans]) >= s - 1e-9) & (lens >= 0.1) &
            (np.array([b for _, b in spans]) <= e + 1e-9)]}) == 1 for _, s, e in SEGS]  # fmt: skip
print(f"đoạn chỉ có một người nói thật: {sum(pure)}/{len(SEGS)}")
assert len(SEGS) == len(ORDER) and all(pure)

Bỏ trước rồi gộp cho đúng 12 đoạn, mỗi đoạn là một lượt nói trọn vẹn của một người. Gộp trước thì nhiều đoạn hơn: một
tiếng tách hay một mảnh vụn bị gán nhãn khác *nằm giữa* hai chữ số của cùng một người, nên nó chặn phép gộp; bỏ nó đi
sau đó thì đã muộn, lượt nói đã bị cắt làm hai. Thứ tự trong báo cáo (bỏ trước, gộp sau) chính là để các mẩu vụn
không cắt vụn những lượt nói dài.

Hình dưới vẽ cả ba lớp trên cùng một trục thời gian: sóng âm, các khoảng có tiếng tô theo cụm (mẩu < 0,1 s gạch chéo),
và các đoạn sau khi gộp, mỗi đoạn ghi nhãn của nó. Dòng dưới cùng là người nói thật.

In [ ]:
COL = {0: BLUE, 1: ORANGE, 2: GREEN, -1: GREY}
lab_name = {}  # màu của mỗi cụm: màu của người nói thật chiếm đa số trong cụm (xám nếu cụm chỉ có mẩu vụn)
for k in set(lab):
    w = true_of[(lab == k) & ok]
    w = w[w >= 0]
    lab_name[k] = COL[int(np.bincount(w).argmax())] if len(w) else GREY
fig, axes = plt.subplots(4, 1, figsize=(10, 4.6), sharex=True, gridspec_kw={"height_ratios": [1.4, 1, 1, 1]})
tt = np.arange(len(rec)) / SR
axes[0].plot(tt, rec, color=GREY, lw=0.4)
axes[0].set_ylabel("sóng âm", fontsize=8)
for (a, b), k, short in zip(spans, lab, lens < 0.1, strict=True):
    axes[1].axvspan(a, b, 0.2, 0.8, color=lab_name[k], alpha=0.5 if short else 0.9, hatch="////" if short else None)
for k, a, b in SEGS:
    axes[2].axvspan(a, b, 0.15, 0.85, color=lab_name[int(k[-2:])], alpha=0.85)
    axes[2].text((a + b) / 2, 0.5, k[-2:], ha="center", va="center", fontsize=7, color="white")
for who, a, b, _ in truth:
    axes[3].axvspan(a, b, 0.2, 0.8, color=COL[who], alpha=0.9)
for t0 in clicks:
    axes[3].axvline(t0, color="k", lw=1)
for ax, name in zip(axes[1:], ("khoảng có tiếng", "sau khi gộp", "người nói thật"), strict=True):
    ax.set_yticks([])
    ax.set_ylabel(name, fontsize=8, rotation=0, ha="right", va="center")
for who, name in enumerate(CAST):
    axes[3].plot([], [], "s", color=COL[who], label=name)
axes[3].plot([], [], "|", color="k", label="tiếng tách")
axes[3].legend(loc="upper center", bbox_to_anchor=(0.5, -0.6), ncol=4, fontsize=8, frameon=False)
axes[3].set_xlabel("thời gian (s)")
vn_axes(fig)
plt.tight_layout()
plt.show()

Mốc một giờ không chạm tới bản ghi 35 giây của ta (mục 12 thử với một mốc ngắn hơn). Trên dữ liệu thật, nó giữ cho
một tập podcast ba giờ không thành một "đơn vị" khổng lồ.

## 6 · Tầng 3: chép lời, rồi kiểm lời

Báo cáo chép lời từng đoạn bằng MOSS-Transcribe-Diarize, một ASR của nhóm vừa chép lời vừa đánh dấu người nói: mỗi
câu bắt đầu bằng một thẻ người nói của *riêng đoạn ấy* ([S1], [S2]…), và có thể có thẻ âm thanh như [music], [laugh]
(tr. 12). Ta giả lập nó: người chấm chữ số chép từng khoảng có tiếng dài từ 0,1 s; khoảng ngắn hơn thành thẻ
[noise]; còn thẻ người nói lấy từ người nói thật, tức một ASR nghe đúng người nói tuyệt đối.

In [ ]:
def transcribe(seg_a, seg_b):
    """Bản chép lời kiểu MOSS-Transcribe-Diarize cho đoạn [seg_a, seg_b]."""
    inside = [i for i, (a, b) in enumerate(spans) if a >= seg_a - 1e-9 and b <= seg_b + 1e-9]
    with torch.no_grad():
        words = asr(judge_input([span_x[i] for i in inside])).argmax(-1).numpy()
    tags, out, cur = {}, [], None
    for i, w in zip(inside, words, strict=True):
        if lens[i] < 0.1:
            out.append("[noise]")
            continue
        tag = tags.setdefault(true_of[i], f"[S{len(tags) + 1}]")
        if tag != cur:
            out.append(tag)
            cur = tag
        out.append(WORDS[w])
    return " ".join(out)


TX = [transcribe(a, b) for _, a, b in SEGS]
truth_words = [" ".join(WORDS[digit[c]] for _, s, e, c in truth if min(e, b) > max(s, a)) for _, a, b in SEGS]
n_ok = sum(re.sub(r"\[[^\]]*\]\s*", "", t) == w for t, w in zip(TX, truth_words, strict=True))
for (k, a, b), tx in list(zip(SEGS, TX, strict=True))[:4]:
    print(f"{k} {vn(a, 2):>6s}–{vn(b, 2):>6s} s: {tx}")
print(f"… chép đúng mọi chữ số ở {n_ok}/{len(SEGS)} đoạn")

Rồi đến kiểm lời. **Ba luật rẻ** chạy trước, để khỏi tốn LLM (tr. 12): lời trống; một cụm từ lặp liền nhau quá 6 lần
(dấu hiệu ASR bị kẹt); sau khi bỏ mọi thẻ trong ngoặc vuông, phần lời còn lại dưới 20% độ dài văn bản (đoạn gần như
toàn nhạc, tiếng động). Sau đó báo cáo dùng **một LLM** chẩn đoán hai lỗi chết (nội dung lặp nguyên câu, câu bị cụt
như có gạch nối ở cuối) rồi sửa ba bước (bỏ thẻ âm thanh, bỏ thẻ người nói không còn lời, sửa khuôn). Ta không có
LLM, nên chỉ làm những phần viết được bằng luật: câu cụt kiểu gạch nối, và hai bước sửa đầu. Cuối cùng là phép
**kiểm một người nói**: bản đã sửa chỉ được có thẻ [S1]. Đây là cùng logic với thẻ 2 trong tab Thực hành của trang.

In [ ]:
def longest_repeat(words, max_len=4):
    """Số lần lặp liền nhau nhiều nhất của một cụm 1–4 từ."""
    best = 1 if words else 0
    for k in range(1, max_len + 1):
        for i in range(len(words) - k + 1):
            n = 1
            while words[i + n * k : i + (n + 1) * k] == words[i : i + k]:
                n += 1
            best = max(best, n)
    return best


def stage3(raw):
    """Trả về (giữ?, lý do, bản đã sửa) theo đúng thứ tự của báo cáo."""
    no_tags = re.sub(r"\[[^\]]*\]", " ", raw)
    words = [re.sub(r"[^\w]", "", w) for w in no_tags.lower().split()]
    words = [w for w in words if w]
    total, speech = len(re.sub(r"\s", "", raw)), len(re.sub(r"\s", "", no_tags))
    if not raw.strip():
        return False, "luật 1: lời trống", ""
    if longest_repeat(words) > 6:
        return False, "luật 2: lặp liền quá 6 lần", ""
    if speech / total < 0.2:
        return False, "luật 3: lời thật dưới 20%", ""
    clean = re.sub(r"\[(?!S\d+\])[^\]]*\]", "", raw)  # sửa 1: bỏ thẻ âm thanh
    clean = re.sub(r"\[S\d+\]\s*(?=\[S\d+\]|$)", "", clean)  # sửa 2: bỏ thẻ người nói không còn lời
    clean = re.sub(r"\s+", " ", clean).strip()
    if re.search(r"[-–]\s*$", clean):
        return False, "LLM: câu bị cụt", clean
    tags = set(re.findall(r"\[S\d+\]", clean))
    if tags != {"[S1]"}:
        return False, f"một người: có {', '.join(sorted(tags))}", clean
    return True, "giữ", clean


EXAMPLES = {
    "sạch": "[S1] Hôm nay mình kể tiếp chương ba. [music]",
    "trống": "  ",
    "lặp": "[S1] được được được được được được được",
    "toàn nhạc": "[S1] [music] [laugh] [music] ừ",
    "cụt": "[S1] Câu này bị cắt ngang ở cu-",
    "hai người": "[S1] Xin chào. [S2] Chào anh.",
}
for name, raw_tx in EXAMPLES.items():
    keep, why, clean = stage3(raw_tx)
    print(f"{name:10s} → {why:28s} {clean!r}" if clean else f"{name:10s} → {why}")
assert [stage3(x)[0] for x in EXAMPLES.values()] == [True, False, False, False, False, False]
res3 = [stage3(x) for x in TX]
print(f"\n{len(SEGS)} đoạn của cuộc trò chuyện: giữ {sum(r[0] for r in res3)}; ví dụ sau khi sửa: {res3[2][2]!r}")

Mọi đoạn của ta đều qua: tầng 2 đã làm đúng việc, nên phép kiểm [S1] không có gì để bắt. Nó là một **lưới an toàn**
cho lỗi của tầng 2. Để thấy nó bắt lỗi, ta làm hỏng bộ tách người nói: hạ ngưỡng gom cụm xuống θ = 0,7, để hai người
có giọng gần nhau bị gom làm một.

In [ ]:
lab_bad = cluster(E_spans, 0.7)
SEGS_BAD = consolidate([(f"SPEAKER_{k:02d}", a, b) for k, (a, b) in zip(lab_bad, spans, strict=True)])
res_bad = [stage3(transcribe(a, b)) for _, a, b in SEGS_BAD]
print(f"θ = 0,7: {len(set(lab_bad[ok]))} cụm cho 3 người, {len(SEGS_BAD)} đoạn; phép kiểm một người loại "
      f"{sum(not r[0] for r in res_bad)} đoạn")  # fmt: skip
for (k, a, b), r in zip(SEGS_BAD, res_bad, strict=True):
    if not r[0]:
        print(f"  {k} {vn(a, 1)}–{vn(b, 1)} s: {r[1]} · {r[2][:70]}")
assert any(not r[0] for r in res_bad)

Với ngưỡng lỏng, tầng 2 gộp những lượt của hai người thành một đoạn dài; ASR nghe ra hai giọng và ghi thêm [S2], nên
tầng 3 loại đoạn ấy. Trên dữ liệu thật, bộ tách người nói và ASR là hai mô hình khác nhau, nên lỗi của cái này có cơ
hội bị cái kia bắt. (Ở đây ASR nghe người nói "tuyệt đối", nên lưới an toàn của ta bắt được mọi lỗi; ASR thật thì
không.)

## 7 · Tầng 4: tốc độ ký tự

Tầng 4 (tr. 13) chấm chất lượng âm thanh trên bản *chưa* khử nhiễu (DNSMOS > 2,8, AudioBox PQ > 6,5), đòi Whisper đoán
ngôn ngữ từ âm thanh trùng với nhãn một LLM đoán từ lời, và kiểm **tốc độ ký tự** r = |x′| / d: số ký tự của lời đã
sửa chia cho độ dài đoạn tính bằng giây. Đoạn quá chậm thường là im lặng hay tiếng nền; đoạn quá nhanh thường là ASR
bịa thêm lời. Khoảng hợp lệ [r_min, r_max] của từng ngôn ngữ lấy từ thống kê của một kho tham chiếu; báo cáo không
công bố các khoảng ấy.

Ta chỉ làm được phần cuối. Kho tham chiếu là các bản ghi huấn luyện của FSDD (số thứ tự từ 10), và khoảng hợp lệ là
từ phân vị 1% đến 99% của chúng, một lựa chọn của riêng notebook này. Ta đếm ký tự không tính dấu cách.

In [ ]:
def char_rate(text, seconds):
    return len(re.sub(r"\[[^\]]*\]|\s", "", text)) / seconds


r_clip = np.array([char_rate(WORDS[d], s) for d, s in zip(digit, dur, strict=True)])
r_min, r_max = np.percentile(r_clip[idx >= 10], [1, 99])
bad = TEST[(r_clip[TEST] < r_min) | (r_clip[TEST] > r_max)]
print(f"khoảng hợp lệ (phân vị 1%–99% của bản ghi huấn luyện): {vn(r_min, 1)}–{vn(r_max, 1)} ký tự/s")
print(f"bị loại trong {len(TEST)} bản ghi kiểm tra: {len(bad)}")
for c in bad[np.argsort(r_clip[bad])][[0, -1]]:
    print(f"  {SPEAKERS[spk[c]]} đọc {WORDS[digit[c]]!r} trong {vn(dur[c], 2)} s: r = {vn(r_clip[c], 1)}")
r_seg = np.array([char_rate(c, b - a) for (_, a, b), (_, _, c) in zip(SEGS, res3, strict=True)])
print(f"12 đoạn của cuộc trò chuyện: r = {vn(r_seg.min(), 1)}–{vn(r_seg.max(), 1)} ký tự/s")

Hình dưới là phân bố r của 3.000 bản ghi (cột), hai vạch là khoảng hợp lệ, và các chấm là 12 đoạn của cuộc trò
chuyện. Bản ghi chậm nhất bị loại thường có một quãng lặng dài ở đầu hay cuối; nghe thử nó ở ô sau.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.hist(r_clip, bins=60, color=BLUE, alpha=0.7, label="3.000 bản ghi FSDD")
for v in (r_min, r_max):
    ax.axvline(v, color=ORANGE, ls="--", lw=1.5)
ax.plot(r_seg, np.full(len(r_seg), 5), "D", color=GREEN, ms=6, label="12 đoạn hội thoại")
ax.text(r_min, ax.get_ylim()[1] * 0.92, " r_min", color=ORANGE, fontsize=8)
ax.text(r_max, ax.get_ylim()[1] * 0.92, " r_max", color=ORANGE, fontsize=8)
ax.set(xlabel="tốc độ ký tự r (ký tự mỗi giây)", ylabel="số bản ghi")
ax.legend(fontsize=8, frameon=False)
vn_axes(fig)
plt.tight_layout()
plt.show()
slow = bad[np.argmin(r_clip[bad])]
play(waves[slow], f"bị loại vì quá chậm: {SPEAKERS[spk[slow]]} đọc {WORDS[digit[slow]]!r}, {vn(dur[slow], 2)} s")

Các đoạn hội thoại chậm hơn hẳn các bản ghi đơn lẻ, vì chúng chứa cả khoảng lặng giữa các chữ số, nhưng vẫn nằm
trong khoảng. Điều đó cho thấy vì sao báo cáo lấy khoảng từ một kho tham chiếu *cùng kiểu* với dữ liệu cần lọc: lấy
khoảng từ những bản ghi một từ mà lọc lời nói liên tục, hay ngược lại, sẽ loại nhầm rất nhiều.

## 8 · Dữ liệu tổng hợp: chọn giọng mẫu

Dữ liệu web cho cặp (âm thanh, lời), nhưng không có **giọng mẫu**: không có gì dạy mô hình nghe một đoạn ngắn rồi đọc
tiếp bằng giọng ấy. Báo cáo dựng cặp (giọng mẫu, đích) từ chính dữ liệu đã lọc (tr. 13–14): gom các đoạn của từng
người nói trong một bản ghi, s₁ … sₙ. Với mỗi đoạn đích sᵢ, từ *mỗi* đoạn khác sⱼ cắt 5 mẩu ngẫu nhiên (đầu và cuối rút
độc lập, dài tối đa 30 s), tức 5(n − 1) ứng viên; chấm cosin giữa vectơ giọng của từng ứng viên và của sᵢ (bằng
WavLM-Large của Seed-TTS-eval), và lấy mẩu cao nhất làm giọng mẫu. Ta dùng vectơ của người chấm giọng thay cho WavLM,
và thêm một điều kiện của riêng notebook: mẩu dài ít nhất 0,25 s, để người chấm có gì mà nghe.

In [ ]:
def crops(a, b, n, rng, max_s=30.0, min_s=0.25):
    out = []
    while len(out) < n:
        s, e = np.sort(rng.uniform(a, b, 2))
        if e - s >= min_s:
            out.append((s, min(e, s + max_s)))
    return out


by_spk = {}
for k, a, b in SEGS:
    by_spk.setdefault(k, []).append((a, b))
rng = np.random.default_rng(1)
PAIRS = []
for k, segs in by_spk.items():
    E_t = embed([rec[int(a * SR) : int(b * SR)] for a, b in segs])
    for i in range(len(segs)):
        cand = [(j, c) for j, (a, b) in enumerate(segs) if j != i for c in crops(a, b, 5, rng)]
        assert len(cand) == 5 * (len(segs) - 1)
        sim = embed([rec[int(s * SR) : int(e * SR)] for _, (s, e) in cand]) @ E_t[i]
        PAIRS.append({"spk": k, "target": segs[i], "cand": cand, "sim": sim, "best": int(sim.argmax())})
print("số đoạn mỗi người:", ", ".join(f"{k}: {len(v)}" for k, v in by_spk.items()))
print(f"mỗi đoạn đích có {len(PAIRS[0]['cand'])} ứng viên = 5 × (4 − 1), mỗi mẩu dài {vn(0.25, 2)}–30 s; "
      f"tổng cộng {len(PAIRS)} cặp")  # fmt: skip
assert all(len(v) == 4 for v in by_spk.values()) and len(PAIRS) == 12

### 🤔 Dự đoán trước
Trong 15 ứng viên, mẩu thắng có phải luôn là mẩu dài nhất không? Mẩu tệ nhất trông thế nào?

In [ ]:
won_longest = 0
for p in PAIRS:
    L_c = np.array([e - s for _, (s, e) in p["cand"]])
    won_longest += int(p["best"] == L_c.argmax())
    p["len"] = L_c
best_sims = np.array([p["sim"][p["best"]] for p in PAIRS])
worst_sims = np.array([p["sim"].min() for p in PAIRS])
print(f"cosin của mẩu thắng: {vn(best_sims.min(), 3)}–{vn(best_sims.max(), 3)}; "
      f"của mẩu tệ nhất: {vn(worst_sims.min(), 3)}–{vn(worst_sims.max(), 3)}")  # fmt: skip
print(f"mẩu thắng là mẩu dài nhất ở {won_longest}/{len(PAIRS)} đoạn đích")
other = []
for p in PAIRS:
    a, b = p["target"]
    e_t = embed([rec[int(a * SR) : int(b * SR)]])[0]
    rival = [x for k, segs in by_spk.items() if k != p["spk"] for x in segs]
    other.append(max(embed([rec[int(s * SR) : int(e * SR)] for s, e in rival]) @ e_t))
print(f"so sánh: cosin cao nhất với một đoạn trọn vẹn của *người khác*: {vn(min(other), 3)}–{vn(max(other), 3)}")

Mẩu thắng thường dài, nhưng không phải lúc nào cũng là mẩu dài nhất; và mẩu tệ nhất có cosin thấp hơn hẳn, đôi khi
thấp hơn cả một đoạn của người khác, vì một mẩu cắt ngẫu nhiên có thể rơi gần hết vào khoảng lặng giữa hai chữ số.
Đó là lý do báo cáo chấm *từng mẩu* thay vì chấm cả đoạn: thứ mô hình nhận lúc dùng là một mẩu ngắn, nên phải chọn
mẩu ngắn nào mang giọng tốt nhất.

Hình dưới là một đoạn đích: mỗi chấm là một ứng viên, trục ngang là độ dài mẩu, trục dọc là cosin; hình dạng chấm
cho biết mẩu cắt từ đoạn nào, và vòng tròn lớn là mẩu thắng. Nghe giọng mẫu và đích ngay sau đó.

In [ ]:
P0 = PAIRS[0]
fig, ax = plt.subplots(figsize=(7, 3))
marks = {}
for (j, _), L_c, s in zip(P0["cand"], P0["len"], P0["sim"], strict=True):
    m = marks.setdefault(j, "osD^"[len(marks)])
    ax.plot(L_c, s, m, color=BLUE, ms=7, alpha=0.8)
ax.plot(P0["len"][P0["best"]], P0["sim"][P0["best"]], "o", ms=16, mfc="none", mec=GREEN, mew=2)
for j, m in marks.items():
    ax.plot([], [], m, color=BLUE, label=f"mẩu từ đoạn {j + 1} của {P0['spk']}")
ax.plot([], [], "o", ms=12, mfc="none", mec=GREEN, mew=2, label="mẩu thắng")
ax.set(xlabel="độ dài mẩu (s)", ylabel="cosin với đoạn đích")
ax.legend(fontsize=8, frameon=False, loc="lower right")
vn_axes(fig)
plt.tight_layout()
plt.show()
j, (s, e) = P0["cand"][P0["best"]]
play(rec[int(s * SR) : int(e * SR)], f"giọng mẫu: {vn(e - s, 2)} s cắt từ đoạn {j + 1}")
a, b = P0["target"]
play(rec[int(a * SR) : int(b * SR)], f"đoạn đích: {vn(b - a, 2)} s")

**Hai bản cho mỗi mẫu.** Để mô hình học cả điều khiển độ dài, mỗi mẫu huấn luyện được ghi hai lần (tr. 14): một bản
có trường `tokens` bằng đúng số token âm thanh của đích, một bản để trường ấy là `None` cho mô hình tự quyết. Bộ token
hóa cho 12,5 khung mỗi giây (tr. 20), nên số token là 12,5 · d.

In [ ]:
FPS = 24000 / (240 * 2 * 2 * 2)  # 12,5 khung mỗi giây (mục 9)
a, b = P0["target"]
text0 = res3[[x[1] for x in SEGS].index(a)][2].removeprefix("[S1] ")
for tokens in (round(FPS * (b - a)), None):
    print({"prompt": f"<{vn(e - s, 2)} s giọng mẫu>", "text": text0, "tokens": tokens})
assert round(FPS * 10) == 125

Báo cáo còn thêm ba loại dữ liệu nhỏ chỉ đổi phần lời, giữ nguyên âm thanh: lời có nhiễu dấu câu và khoảng trắng
(người dùng thật gõ "Hello??!!   are you there"), lời viết bằng phiên âm (pinyin có thanh điệu cho tiếng Trung, IPA
giữa hai gạch chéo cho tiếng Anh), và những câu chỉ một từ. Chúng là phép biến đổi chuỗi, nên ta không lặp lại ở đây.

## 9 · Bộ token hóa: đếm

MOSS-Audio-Tokenizer (tr. 4–7) là bộ token hóa CAT mà Chuyên đề 05 dựng lại ở bản thu nhỏ. Ở kích thước thật: 24.000
mẫu mỗi giây, ghép mảng 240 rồi ba lần 2, nên mỗi khung là 1.920 mẫu; RVQ 32 lớp, mỗi lớp 1.024 mã.

In [ ]:
SR_TOK, PATCHES, NQ, CODES = 24000, (240, 2, 2, 2), 32, 1024
hop = math.prod(PATCHES)
ladder = [SR_TOK]
for p in PATCHES:
    ladder.append(ladder[-1] / p)
bits = math.log2(CODES)
print(" → ".join(vn(v, 1 if v % 1 else 0) for v in ladder), "vị trí mỗi giây")
print(f"mỗi khung {vn(hop, 0)} mẫu; mỗi lớp {vn(FPS * bits, 0)} bit/s; đủ {NQ} lớp {vn(FPS * bits * NQ, 0)} bit/s")
print(f"giai đoạn 1 của bộ token hóa: 5 giờ âm thanh mỗi batch 1.536 ví dụ = {vn(5 * 3600 / 1536, 2)} s mỗi ví dụ")
assert hop == 1920 and ladder[-1] == FPS == 12.5 and FPS * bits == 125 and FPS * bits * NQ == 4000

Bộ token hóa học riêng, trước mô hình TTS, và học trên âm thanh rộng hơn (cả nhạc và tiếng động): 520.000 bước không
có GAN rồi 500.000 bước có GAN (tr. 6–7). Cách nó học, và vì sao *quantizer dropout* cho nó dùng được ở mọi số lớp,
là nội dung của Chuyên đề 05.

## 10 · 33 kênh token, và delay pattern

Ở mỗi bước, cả hai mô hình của báo cáo đoán **33 kênh** (tr. 7–8): kênh 0 là chữ (ở những bước chỉ có chữ) hoặc một
ký hiệu đệm (ở những bước âm thanh), và 32 kênh âm thanh, mỗi kênh một lớp RVQ. Vấn đề: 32 token của cùng một khung
không độc lập, lớp 2 sửa lỗi của lớp 1. Nếu đoán cả 32 cùng lúc từ một trạng thái, lớp 2 không biết lớp 1 đã chọn gì.

**Delay pattern** (MusicGen; tr. 8–9) dịch lớp j đi j − 1 bước: ã_{j,t} = a_{j,t−(j−1)}. Ở bước t, mô hình đoán lớp 1
của khung t, lớp 2 của khung t − 1, …: khi đến lượt lớp j của một khung, các lớp trước của khung ấy đã nằm trong đầu
vào. Đầu vào mỗi bước là *tổng* embedding của 32 lớp, và một mặt nạ m che các ô trống do phép dịch tạo ra.

Ta dùng token thật: chữ số "three" ở 32 lớp do CAT-TTS thu nhỏ của Chuyên đề 05 sinh ra (lưu trong tệp dữ liệu), cộng
năm token chữ t, h, r, e, e (ở đây mỗi ký tự là một token, cho dễ nhìn).

In [ ]:
SH = W["show"]["0"]
assert SH["k"][0] == NQ and SH["target"][0] == 3
A = torch.tensor(SH["codes"][0][0])  # (T khung, 32 lớp)
T = A.shape[0]
PAD = -1


def delay(A):
    """(T, Nq) → (T + Nq − 1, Nq) và mặt nạ: lớp j (đếm từ 0) dịch đi j bước."""
    T_, nq = A.shape
    D_ = torch.full((T_ + nq - 1, nq), PAD)
    m = torch.zeros(T_ + nq - 1, nq, dtype=torch.bool)
    for j in range(nq):
        D_[j : j + T_, j] = A[:, j]
        m[j : j + T_, j] = True
    return D_, m


def undelay(D_, T_):
    return torch.stack([D_[j : j + T_, j] for j in range(D_.shape[1])], 1)


Dl, mask = delay(A)
assert torch.equal(undelay(Dl, T), A), "phép dịch ngược phải trả lại đúng ma trận token"
step_of = torch.stack([torch.arange(T) + j for j in range(NQ)], 1)  # bước sinh ra lớp j của khung f
assert (step_of[:, 1:] > step_of[:, :-1]).all(), "trong một khung, lớp sau luôn được sinh sau lớp trước"
print(
    f"{T} khung × {NQ} lớp → {Dl.shape[0]} bước × {NQ} kênh âm thanh; ô bị che: {int((~mask).sum())} = {NQ} × {NQ - 1}"
)
print("khung 0: lớp 1, 2, 3, …, 32 được sinh ở các bước", step_of[0, :3].tolist(), "…", step_of[0, -1].item())
assert int((~mask).sum()) == NQ * (NQ - 1)

Hình dưới là chuỗi 33 kênh mà Transformer đọc: 5 bước chữ ở đầu (kênh 0 là chữ, các kênh âm thanh bị che), rồi
T + 31 bước âm thanh (kênh 0 là đệm). Mỗi màu là một mã; ô gạch chéo là ô bị che, không tính vào mất mát. Hàng lớp 1
bắt đầu ngay, hàng lớp 32 bắt đầu muộn 31 bước, nên các ô thật tạo thành một dải chéo.

In [ ]:
TEXT = "three"
n_text = len(TEXT)
codes = np.full((NQ + 1, n_text + Dl.shape[0]), np.nan)  # chỉ các ô âm thanh thật có màu
codes[1:, n_text:] = np.where(mask.T.numpy(), Dl.T.numpy() % 97 / 97, np.nan)
hidden = np.zeros(codes.shape, bool)
hidden[1:, :n_text] = True  # bước chữ: các kênh âm thanh bị che
hidden[1:, n_text:] = ~mask.T.numpy()
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(codes, aspect="auto", cmap="viridis", interpolation="nearest")
ys, xs = np.nonzero(hidden)
ax.add_collection(PatchCollection([Rectangle((x - 0.5, y - 0.5), 1, 1) for y, x in zip(ys, xs, strict=True)],
                                  facecolor="none", edgecolor=GREY, hatch="////", linewidth=0))  # fmt: skip
for x in range(codes.shape[1]):
    is_text = x < n_text
    ax.add_patch(Rectangle((x - 0.5, -0.5), 1, 1, facecolor=GREEN if is_text else "#d9dde3", edgecolor="white"))
    if is_text:
        ax.text(x, 0, TEXT[x], ha="center", va="center", fontsize=7, color="white")
ax.text(n_text + 0.5, 0, "đệm (vẫn là đích để đoán)", va="center", fontsize=7)
ax.set_yticks([0, 1, 8, 16, 32], ["kênh 0 · chữ/đệm", "lớp 1", "lớp 8", "lớp 16", "lớp 32"], fontsize=8)
ax.set_xlabel("bước của Transformer")
ax.axvline(n_text - 0.5, color="k", lw=1)
vn_axes(fig)
plt.tight_layout()
plt.show()

### 🤔 Dự đoán trước
Một câu 10 giây là 125 khung. Với delay pattern, Transformer lớn phải chạy bao nhiêu bước để sinh hết âm thanh? Với
Local Transformer (mô hình nhỏ chạy bên trong từng khung, tr. 9–10) thì Transformer lớn chạy bao nhiêu bước, và mô
hình nhỏ bao nhiêu bước? Khung đầu tiên đủ 32 token sau bao nhiêu bước?

In [ ]:
def steps_delay(T_, nq=NQ):
    return T_ + nq - 1


def steps_local(T_, nq=NQ):
    return T_, T_ * (nq + 1)  # bước lớn; bước nhỏ (33 kênh mỗi khung)


for frames in (T, 100, 125, 750):
    big, small = steps_local(frames)
    print(f"{frames:4d} khung ({vn(frames / FPS, 1):>4s} s): delay {steps_delay(frames):4d} bước lớn, "
          f"ô che chiếm {pct(NQ * (NQ - 1) / (steps_delay(frames) * NQ))} · local {big} bước lớn + "
          f"{vn(small, 0)} bước nhỏ")  # fmt: skip
assert steps_delay(100) == 131 and steps_delay(125) == 156 and steps_delay(1) == 32

Delay pattern chỉ thêm 31 bước, bất kể câu dài bao nhiêu: một câu 10 giây là 156 bước thay vì 125, và tỉ lệ ô bị che
giảm dần khi câu dài ra. Mỗi bước chỉ cần một lần chạy Transformer lớn rồi 33 phép chiếu, nên nó đơn giản và dễ mở
rộng; báo cáo dùng nó cho mô hình chính 8 tỉ tham số. Giá phải trả: khung đầu tiên chỉ đủ 32 token sau 32 bước.
Local Transformer không dịch gì: Transformer lớn chạy đúng T bước, rồi một mô hình nhỏ sinh lần lượt 33 kênh của
khung kế tiếp, mỗi kênh nhìn thấy các kênh trước nó *trong cùng khung*. Nhiều bước nhỏ hơn hẳn, nhưng mỗi bước nhỏ rẻ;
và trong báo cáo, bản local 1,7 tỉ giữ giọng tốt hơn bản delay 8 tỉ trên Seed-TTS-eval (Bảng 3, tr. 19).

**Mất mát có trọng số.** Cả hai mô hình dùng cùng một entropy chéo có trọng số theo kênh (tr. 8):
λ = (1, 3, 3, 3, 2, 2, 2, 1, …, 1). Kênh chữ/đệm và 26 lớp cuối có trọng số 1; ba lớp đầu 3, ba lớp kế 2. Ô dưới tính
mất mát trên 32 kênh âm thanh của một mô hình "chưa học gì" (mọi mã có xác suất như nhau) cho chuỗi delay ở trên,
có và không có mặt nạ, và phần trọng số mà 6 lớp đầu gánh.

In [ ]:
LAM = torch.tensor([1, 3, 3, 3, 2, 2, 2] + [1] * 26, dtype=torch.float32)
assert len(LAM) == NQ + 1 and LAM.sum() == 42
logits = torch.zeros(Dl.shape[0], NQ, CODES)  # đoán đều mọi mã
ce = F.cross_entropy(logits.reshape(-1, CODES), Dl.clamp_min(0).reshape(-1), reduction="none").view(Dl.shape)
per_layer = (ce * mask).sum(0)  # mất mát mỗi lớp âm thanh, chỉ trên ô thật
loss = (LAM[1:] * per_layer).sum()
print(f"mỗi ô: −log(1/1.024) = {vn(math.log(CODES), 3)}; mỗi lớp có {T} ô thật trên {Dl.shape[0]} ô")
print(f"mất mát có mặt nạ: {vn(loss, 1)} = {vn(math.log(CODES), 3)} × {T} × {vn(LAM[1:].sum(), 0)}")
print(f"không có mặt nạ, các ô trống cũng bị tính: {vn((LAM[1:] * ce.sum(0)).sum(), 1)}")
print(f"6 lớp đầu là {pct(6 / 33)} số kênh nhưng gánh {pct(LAM[1:7].sum() / LAM.sum())} tổng trọng số")
assert abs(loss - math.log(CODES) * T * 41) < 1e-2

Không có mặt nạ, mô hình bị phạt vì không đoán được những ô *không có token nào*. Với trọng số λ, các lớp đầu (mang
phần thô, quan trọng nhất của âm thanh: Chuyên đề 05 thấy 1 lớp đã đủ nghe ra chữ số) được chú ý gấp 2–3 lần.

## 11 · Bốn pha và lịch tốc độ học

Tiền huấn luyện chia bốn pha (Bảng 1, tr. 15–16), theo kiểu **khởi động–giữ–giảm** (WSD, như MiniCPM):

| Pha | Chuỗi dài nhất | Tốc độ học | Dữ liệu |
|---|---|---|---|
| 1 | 32k | khởi động lên 2·10⁻⁴ rồi giữ | chỉ dữ liệu đã lọc $\mathcal{D}_{\text{basic}}$ |
| 2 | 32k | giữ 2·10⁻⁴ | mọi tập; giọng mẫu $\mathcal{D}_{\text{clone}}$ được lấy nhiều hơn hẳn |
| 3 | 32k | giảm tuyến tính 2·10⁻⁴ → 2·10⁻⁶ | mọi tập, tỉ lệ bình thường |
| 4 | 64k | giữ 2·10⁻⁶ | thêm dữ liệu dài, mở chuỗi lên 64k |

Báo cáo không công bố độ dài các pha. Ta vẽ lịch với bốn pha dài bằng nhau và phần khởi động 15% của pha 1, giống thẻ
5 trong tab Thực hành; đó là giả định, không phải số của báo cáo.

In [ ]:
LR_PEAK, LR_END = 2e-4, 2e-6


WARM = 0.15  # phần khởi động trong pha 1; báo cáo không nêu


def lr_at(u, peak=LR_PEAK, end=LR_END, lengths=(1, 1, 1, 1), warm=WARM):
    """Tốc độ học tại tiến trình u ∈ [0, 1] của lịch bốn pha."""
    e = np.cumsum([0, *lengths]) / sum(lengths)
    if u < e[1] * warm:
        return peak * u / (e[1] * warm)
    if u < e[2]:
        return peak
    if u < e[3]:
        return peak + (end - peak) * (u - e[2]) / (e[3] - e[2])
    return end


print(f"đỉnh {sci(LR_PEAK)}, cuối {sci(LR_END)}: giảm {vn(LR_PEAK / LR_END, 0)} lần, và chỉ giảm trong pha 3")
print(f"giả định của notebook: bốn pha dài bằng nhau, khởi động trong {pct(WARM, 0)} đầu của pha 1")
assert round(LR_PEAK / LR_END) == 100 and lr_at(0.5) == LR_PEAK and lr_at(0.9) == LR_END

Vì sao giữ tốc độ học cao suốt hai pha, rồi mới giảm? Với tốc độ học cao, mỗi bước cập nhật vẫn nhảy quanh đáy thay
vì chạm đáy, nên mất mát dừng ở một mức "nhiễu". Phần giảm gom những bước nhảy ấy lại. Ta thử trên một bài toán đủ
nhỏ để chạy trong vài giây: hồi quy softmax đoán chữ số từ 80 đặc trưng log-mel (trung bình và độ lệch chuẩn của 40
dải) của mỗi bản ghi FSDD, Adam, batch 16, 2.000 bước, 3 seed. Hai lịch có cùng đỉnh 0,03: giữ nguyên suốt, hoặc bốn
pha như trên (đỉnh/100 ở cuối).

In [ ]:
Xf = np.zeros((len(waves), 10240), np.float32)
for b, x in enumerate(waves):
    x = x[:10240]
    Xf[b, : len(x)] = x * 0.1 / np.sqrt(np.mean(x**2))
with torch.no_grad():
    M = logmel(torch.tensor(Xf))
    act = (M.max(1).values > -3).float()[:, None]  # chỉ các khung có tiếng
    mu = (M * act).sum(-1) / act.sum(-1)
    sd = (((M - mu[..., None]) ** 2) * act).sum(-1).div(act.sum(-1)).sqrt()
    feats = torch.cat([mu, sd], 1)
TRN = np.where(idx >= 5)[0]
feats = (feats - feats[TRN].mean(0)) / feats[TRN].std(0)
y_d = torch.tensor(digit)


STEPS, PEAK, BS = 2000, 0.03, 16


def run(schedule, seed, steps=STEPS, peak=PEAK, bs=BS, lengths=(1, 1, 1, 1)):
    g = torch.Generator().manual_seed(seed)
    Wl, bl = torch.zeros(80, 10, requires_grad=True), torch.zeros(10, requires_grad=True)
    opt = torch.optim.Adam([Wl, bl], lr=peak)
    curve = []
    for k in range(steps):
        lr = lr_at(k / steps, peak, peak / 100, lengths) if schedule == "wsd" else peak
        for pg in opt.param_groups:
            pg["lr"] = lr
        i = torch.tensor(TRN)[torch.randint(len(TRN), (bs,), generator=g)]
        loss = F.cross_entropy(feats[i] @ Wl + bl, y_d[i])
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (k + 1) % 50 == 0:
            with torch.no_grad():
                curve.append(float(F.cross_entropy(feats[TRN] @ Wl + bl, y_d[TRN])))
    with torch.no_grad():
        acc = float(((feats[TEST] @ Wl + bl).argmax(1) == y_d[TEST]).float().mean())
    return np.array(curve), acc


RUNS = {s: [run(s, seed) for seed in range(3)] for s in ("const", "wsd")}
print(
    f"Adam, batch {BS}, {vn(STEPS, 0)} bước, 3 seed; đỉnh {vn(PEAK, 2)} (lịch bốn pha kết thúc ở {vn(PEAK / 100, 4)})"
)
for s, name in (("const", "giữ nguyên"), ("wsd", "bốn pha")):
    c = np.array([r[0] for r in RUNS[s]])
    accs = [r[1] for r in RUNS[s]]
    print(f"{name:10s} mất mát huấn luyện ở 50%: {vn(c[:, 19].min(), 3)}–{vn(c[:, 19].max(), 3)} · ở 75%: "
          f"{vn(c[:, 29].min(), 3)}–{vn(c[:, 29].max(), 3)} · cuối: {vn(c[:, -1].min(), 3)}–{vn(c[:, -1].max(), 3)} · "
          f"đúng trên tập kiểm tra {pct(min(accs))}–{pct(max(accs))}")  # fmt: skip
cw, cc = np.array([r[0] for r in RUNS["wsd"]]), np.array([r[0] for r in RUNS["const"]])
assert cw[:, -1].max() < cc[:, -1].min(), "bốn pha phải kết thúc thấp hơn giữ nguyên, ở cả 3 seed"

### 🤔 Dự đoán trước
Hình dưới vẽ lịch tốc độ học (trên) và mất mát huấn luyện của hai lịch (dưới). Trước khi xem: khi tốc độ học bắt đầu
giảm ở đầu pha 3 (50% tiến trình), mất mát của lịch bốn pha đi lên, đứng yên hay tụt xuống?

In [ ]:
u = np.arange(1, 41) * 50 / STEPS
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True, gridspec_kw={"height_ratios": [1, 1.6]})
uu = np.linspace(0, 1, 401)
a1.plot(uu, [lr_at(v, PEAK, PEAK / 100) for v in uu], color=BLUE, lw=2, label="bốn pha")
a1.plot(uu, np.full_like(uu, PEAK), color=ORANGE, lw=2, ls="--", label="giữ nguyên")
a1.set(ylabel="tốc độ học", ylim=(0, 1.23 * PEAK))
a1.legend(fontsize=8, frameon=False)
for s, col, ls in (("wsd", BLUE, "-"), ("const", ORANGE, "--")):
    for c, _ in RUNS[s]:
        a2.plot(u, c, color=col, ls=ls, lw=1.2, alpha=0.85)
a2.set(yscale="log", xlabel="tiến trình huấn luyện", ylabel="mất mát huấn luyện")
for ax in (a1, a2):
    for x in (0.25, 0.5, 0.75):
        ax.axvline(x, color=GREY, lw=0.8, ls=":")
for i in range(4):
    a1.text(0.125 + 0.25 * i, 1.1 * PEAK, f"pha {i + 1}", ha="center", fontsize=8)
vn_axes(fig)
plt.tight_layout()
plt.show()

Mất mát của lịch bốn pha tụt xuống trong pha 3, khi tốc độ học giảm dần, rồi nằm yên ở pha 4; lịch giữ nguyên thì dao
động quanh một mức cao hơn đến cuối. Đó là điều báo cáo gọi là pha 3 "củng cố chất lượng": phần lớn cái lợi cuối cùng
đến trong đoạn giảm. Lợi ích thực tế của WSD là ta có thể giữ tốc độ học cao bao lâu tùy thích (thêm dữ liệu, như
$\mathcal{D}_{\text{clone}}$ ở pha 2) rồi mới giảm, thay vì phải định trước tổng số bước như lịch cosin. Bài toán của
ta lồi và nhỏ, nên nó chỉ minh họa hình dạng của hiện tượng, không minh họa lý do báo cáo chọn đưa giọng mẫu vào sớm
hay mở chuỗi 64k muộn.

## 12 · Thử thay đổi

### Ngưỡng gom cụm θ
θ quá cao tách một người làm nhiều nhãn; θ quá thấp gom nhiều người làm một. Ô dưới quét θ và đếm số cụm (chỉ tính
khoảng ≥ 0,1 s), số đoạn sau khi gộp và số đoạn bị phép kiểm một người loại.

In [ ]:
for theta in (0.6, 0.7, 0.8, 0.85, 0.9, 0.94, 0.96, 0.98):
    lb = cluster(E_spans, theta)
    sg = consolidate([(f"SPEAKER_{k:02d}", a, b) for k, (a, b) in zip(lb, spans, strict=True)])
    dropped = sum(not stage3(transcribe(a, b))[0] for _, a, b in sg)
    print(f"θ = {vn(theta, 2)}: {len(set(lb[ok])):2d} cụm, {len(sg):2d} đoạn, tầng 3 loại {dropped}")

### τ_min và mốc chặn
Báo cáo chặn ở một giờ; với bản ghi 35 giây của ta, thử chặn ở 9 giây (tính từ đầu đoạn đầu tiên) để thấy đoạn cuối
bị cắt. Rồi thử nâng τ_min lên 0,5 s: nhiều chữ số ngắn hơn nửa giây, nên chúng bị bỏ như mảnh vụn.

In [ ]:
sg = consolidate(D, cap=9.0)
k, a, b = sg[-1]
print(f"chặn ở 9 s: {len(sg)} đoạn; đoạn cuối {k} {vn(a, 2)}–{vn(b, 2)} s (không chặn thì đến "
      f"{vn(SEGS[len(sg) - 1][2], 2)} s)")  # fmt: skip
sg = consolidate(D, tau_min=0.5)
print(f"τ_min = 0,5 s: chỉ {sum(e - s >= 0.5 for _, s, e in D)}/{len(D)} khoảng đủ dài → {len(sg)} đoạn: "
      + ", ".join(f"{k} {vn(a, 2)}–{vn(b, 2)} s ({vn(b - a, 1)} s)" for k, a, b in sg))  # fmt: skip
k, a, b = sg[0]
inside = [CAST[w] for w in sorted({w for w, s, e, _ in truth if s >= a and e <= b})]
print(f"đoạn ấy chứa lời của {', '.join(inside)}; tầng 3: {stage3(transcribe(a, b))[1]}")

Với τ_min = 0,5 s, chỉ năm chữ số của george đủ dài để được giữ. Luật gộp không xét khoảng lặng, nên năm khoảng cùng
nhãn ấy thành **một** đoạn dài, nuốt trọn các lượt của lucas và theo nằm giữa chúng. Luật "gộp mọi đoạn liền nhau cùng
nhãn" chỉ an toàn khi lượt của người khác còn nằm trong chuỗi để chặn phép gộp; τ_min nhỏ (0,1 s) giữ điều ấy. Ở đây
tầng 3 vẫn bắt được đoạn hỏng, vì ASR của ta nghe ra ba giọng.

### Ít mẩu hơn cho mỗi đoạn
Báo cáo cắt 5 mẩu từ mỗi đoạn khác. Với 1 mẩu, mẩu thắng tệ hơn bao nhiêu? Sửa `n_crops` rồi chạy lại.

In [ ]:
n_crops = 1
rng = np.random.default_rng(1)
few = []
for k, segs in by_spk.items():
    E_t = embed([rec[int(a * SR) : int(b * SR)] for a, b in segs])
    for i in range(len(segs)):
        cand = [c for j, (a, b) in enumerate(segs) if j != i for c in crops(a, b, n_crops, rng)]
        few.append(max(embed([rec[int(s * SR) : int(e * SR)] for s, e in cand]) @ E_t[i]))
print(f"{n_crops} mẩu mỗi đoạn: cosin của mẩu thắng {vn(min(few), 3)}–{vn(max(few), 3)} "
      f"(5 mẩu: {vn(best_sims.min(), 3)}–{vn(best_sims.max(), 3)})")  # fmt: skip

### Độ dài các pha
Báo cáo không công bố độ dài pha. Thử một pha 3 ngắn (giảm nhanh) và một pha 3 dài, với cùng tổng số bước.

In [ ]:
def ratio(lengths):
    return "(" + "; ".join(vn_tick(v) for v in lengths) + ")"


print(f"độ dài pha {ratio((1, 1, 1, 1))}: mất mát cuối {vn(cw[:, -1].min(), 3)}–{vn(cw[:, -1].max(), 3)} (ở trên)")
for lengths in ((1, 1, 0.3, 1), (1, 1, 2.5, 0.2)):
    finals = [run("wsd", seed, lengths=lengths)[0][-1] for seed in range(3)]
    print(f"độ dài pha {ratio(lengths)}: mất mát cuối {vn(min(finals), 3)}–{vn(max(finals), 3)}")

Trên bài toán này, pha 3 càng dài thì mất mát cuối càng thấp: với cùng tổng số bước, bước giảm dần đáng giá hơn bước
giữ ở đỉnh hay ở đáy. Báo cáo không cho biết nó chia thời gian thế nào.

## 13 · Giới hạn

- **Không có mô hình lớn nào chạy ở đây.** Khử nhiễu, chấm DNSMOS và AudioBox PQ, LLM chẩn đoán và sửa lời, Whisper
  đoán ngôn ngữ, WavLM-Large, bộ token hóa 1,6 tỉ tham số (tr. 4) và hai mô hình 8 tỉ và 1,7 tỉ: ta chỉ dựng lại các
  phép tính xung quanh chúng. Hai người chấm của Chuyên đề 05 thay cho ASR và vectơ giọng, và chúng chỉ biết 10 chữ số
  và 6 giọng.
- **Dữ liệu dựng sẵn.** Cuộc trò chuyện được ghép từ những bản ghi một từ, cùng mức âm, nền gần như im lặng. Không có
  chồng tiếng, nhạc nền hay tiếng vang, những thứ làm việc tách người nói thật khó hơn nhiều.
- **ASR nghe người nói tuyệt đối.** Thẻ [S1]/[S2] của bản chép lời lấy từ người nói thật, nên phép kiểm một người ở
  mục 6 bắt mọi lỗi của tầng 2; một ASR thật thì không.
- **Những chỗ báo cáo không nói rõ:** cách đọc bước chia cho biên độ lớn nhất, giá trị ε, khoảng [r_min, r_max], độ
  dài các pha, độ dài khởi động. Notebook chọn một giá trị cho mỗi chỗ và ghi rõ là lựa chọn của mình.
- **Bài toán ở mục 11 lồi và nhỏ.** Nó cho thấy hình dạng của hiệu ứng WSD, không cho thấy nó lớn đến đâu ở một mô
  hình 8 tỉ tham số.

## Tóm tắt

- **Tầng 1:** độ lợi clip(−20 − L, −3, 3) dB gần như không thu hẹp độ chênh ~38 dB của FSDD; nếu luôn chia cho max |x|
  thì độ lợi ấy bị triệt tiêu, và chính phép chia làm mức âm dồn lại. Báo cáo không nói rõ cách nào.
- **Tầng 2:** dò tiếng bằng năng lượng, gom cụm vectơ giọng ở θ = 0,9, rồi bỏ mẩu < 0,1 s *trước khi* gộp: 12 lượt nói
  thành đúng 12 đoạn một người. Gộp trước thì các mẩu vụn cắt lượt nói làm nhiều đoạn hơn.
- **Tầng 3:** ba luật rẻ, phần dễ của LLM, rồi phép kiểm chỉ [S1], một lưới an toàn cho lỗi gộp của tầng 2.
- **Tầng 4:** tốc độ ký tự loại những bản ghi quá chậm (nhiều im lặng) hay quá nhanh; khoảng hợp lệ phải lấy từ một kho
  cùng kiểu.
- **Giọng mẫu:** 5(n − 1) ứng viên, chấm từng mẩu bằng cosin; mẩu thắng không phải lúc nào cũng dài nhất.
- **Mô hình:** 33 kênh mỗi bước; delay pattern thêm đúng 31 bước (156 cho 125 khung) và che 32 × 31 ô; Local
  Transformer chạy 125 bước lớn và 4.125 bước nhỏ; 6 lớp đầu gánh 35,7% tổng trọng số λ.
- **Bốn pha:** giữ tốc độ học cao, rồi giảm 100 lần chỉ trong pha 3; trên bài toán nhỏ, mất mát tụt xuống trong pha 3
  và kết thúc thấp hơn lịch giữ nguyên ở cả 3 seed.

## Câu hỏi ôn tập

1. Một bản ghi ở −14 dBFS có đỉnh 0,5. Sau tầng 1, mức RMS là bao nhiêu theo cách đọc A? Theo cách đọc B?
2. Vì sao báo cáo chấm DNSMOS và AudioBox PQ trên bản *chưa* khử nhiễu, dù đã khử nhiễu ở tầng 1?
3. Một người có 5 đoạn trong một bản ghi. Với một đoạn đích, có bao nhiêu ứng viên giọng mẫu? Vì sao không lấy luôn
   cả một đoạn khác làm giọng mẫu?
4. Với delay pattern, một câu 8 giây cần bao nhiêu bước của Transformer? Bao nhiêu ô bị che?
5. Bỏ mặt nạ thì mất mát của mô hình "chưa học gì" ở mục 10 tăng lên. Vì sao một mô hình học với mất mát không mặt nạ
   lại có hại, chứ không chỉ tốn thêm tính toán?
6. Báo cáo đưa $\mathcal{D}_{\text{clone}}$ vào ở pha 2, khi tốc độ học còn cao, rồi trả tỉ lệ về bình thường ở pha
   3. Theo báo cáo, điều
   gì xảy ra nếu lấy nhiều giọng mẫu quá lâu?

<details><summary>Gợi ý đáp án</summary>

1. L = −14, nên độ lợi bị chặn ở −3 dB, mức thành −17 dBFS và đỉnh thành 0,5 · 10^(−3/20) ≈ 0,35. Cách đọc A chia cho
   0,35: mức thành −17 − 20 log₁₀ 0,35 ≈ −8 dBFS (tức −14 − 20 log₁₀ 0,5, như thể không có độ lợi). Cách đọc B: đỉnh
   không vượt 1, nên giữ −17 dBFS.
2. Bộ khử nhiễu có thể làm điểm chất lượng đẹp lên: điểm khi ấy phản ánh bộ khử nhiễu chứ không phải bản ghi gốc. Khử
   nhiễu ở tầng 1 chỉ để bước tách người nói chính xác hơn, không phải tín hiệu huấn luyện.
3. 5 · (5 − 1) = 20. Lúc dùng, mô hình nhận một mẩu ngắn tùy ý; chấm từng mẩu chọn mẩu ngắn nào mang giọng tốt nhất,
   và độ dài, vị trí ngẫu nhiên (tối đa 30 s) dạy mô hình chịu được giọng mẫu dài ngắn khác nhau.
4. 8 giây là 100 khung, nên 100 + 31 = 131 bước; số ô bị che luôn là 32 · 31 = 992.
5. Mô hình sẽ được dạy đoán những ô không có token thật (ví dụ đoán "đệm" ở đó), tức học một quy luật của cách xếp chứ
   không phải của âm thanh, và phần trọng số ấy lấy mất sự chú ý khỏi các ô thật.
6. Mô hình thiên về chép giọng mẫu (*prompt copying*), và các khả năng khác (TTS thường đa ngôn ngữ, đọc từ đơn, chịu
   được lời nhiễu) bị lấn át (tr. 15).
</details>

## Tìm hiểu thêm

- SII-OpenMOSS (2026), *MOSS-TTS Technical Report*: [arXiv:2603.18090](https://arxiv.org/abs/2603.18090); mã và mô
  hình: [OpenMOSS/MOSS-TTS](https://github.com/OpenMOSS/MOSS-TTS).
- Gong và cộng sự (2026), *MOSS-Audio-Tokenizer*: [arXiv:2602.10934](https://arxiv.org/abs/2602.10934).
- Copet và cộng sự (2023), *Simple and Controllable Music Generation* (MusicGen, delay pattern):
  [arXiv:2306.05284](https://arxiv.org/abs/2306.05284).
- Défossez và cộng sự (2024), *Moshi* (RQ-Transformer theo thời gian và theo chiều sâu):
  [arXiv:2410.00037](https://arxiv.org/abs/2410.00037).
- Hu và cộng sự (2024), *MiniCPM* (lịch tốc độ học WSD): [arXiv:2404.06395](https://arxiv.org/abs/2404.06395).
- Han và cộng sự (2025), *DiariZen*: [arXiv:2409.09408](https://arxiv.org/abs/2409.09408).
- Radford và cộng sự (2022), *Whisper*: [arXiv:2212.04356](https://arxiv.org/abs/2212.04356).
- Reddy, Gopal & Cutler (2022), *DNSMOS P.835*: [arXiv:2110.01763](https://arxiv.org/abs/2110.01763).
- Free Spoken Digit Dataset: [github.com/Jakobovski/free-spoken-digit-dataset](https://github.com/Jakobovski/free-spoken-digit-dataset).

**Quay lại:** [Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/) ·
[Chuyên đề 05 · Token hóa âm thanh](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-hoa-am-thanh/).

In [ ]:
import platform

import matplotlib

print(
    f"Python {platform.python_version()} · numpy {np.__version__} · torch {torch.__version__} · "
    f"matplotlib {matplotlib.__version__}"
)